In [8]:
import pandas as pd
import pandapower as pp
import numpy as np
from os import path

# --- File Loading ---
# NOTE: This path will need to be adjusted if you run this outside the immediate 
# environment where the file was uploaded. In this environment, it should work.
data_file = "IEEE_118_bus_20k_data.csv"

# Load the raw dataset
try:
    df_raw = pd.read_csv(data_file)
    print(f"Data loaded successfully. Rows: {len(df_raw)}, Columns: {len(df_raw.columns)}")
except Exception as e:
    print(f"Error loading file: {e}")

# --- Pandapower Setup ---
# Create the standard IEEE 118 bus network model
net = pp.networks.case118()
print(f"Pandapower network 'net' created. Buses: {len(net.bus)}")

Data loaded successfully. Rows: 20001, Columns: 473
Pandapower network 'net' created. Buses: 118


In [9]:
print(df_raw.columns)

Index(['VGM1', 'VGM4', 'VGM6', 'VGM8', 'VGM10', 'VGM12', 'VGM15', 'VGM18',
       'VGM19', 'VGM24',
       ...
       'QL101', 'QL102', 'QL106', 'QL108', 'QL109', 'QL114', 'QL115', 'QL117',
       'QL118', 'Label'],
      dtype='object', length=473)


In [10]:
bus_indices = net.bus.index 
P_cols = [f"PG_{i}" for i in bus_indices]
Q_cols = [f"QG_{i}" for i in bus_indices]
print(bus_indices)
print(P_cols)
print(Q_cols)

Index([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,
       ...
       108, 109, 110, 111, 112, 113, 114, 115, 116, 117],
      dtype='int64', length=118)
['PG_0', 'PG_1', 'PG_2', 'PG_3', 'PG_4', 'PG_5', 'PG_6', 'PG_7', 'PG_8', 'PG_9', 'PG_10', 'PG_11', 'PG_12', 'PG_13', 'PG_14', 'PG_15', 'PG_16', 'PG_17', 'PG_18', 'PG_19', 'PG_20', 'PG_21', 'PG_22', 'PG_23', 'PG_24', 'PG_25', 'PG_26', 'PG_27', 'PG_28', 'PG_29', 'PG_30', 'PG_31', 'PG_32', 'PG_33', 'PG_34', 'PG_35', 'PG_36', 'PG_37', 'PG_38', 'PG_39', 'PG_40', 'PG_41', 'PG_42', 'PG_43', 'PG_44', 'PG_45', 'PG_46', 'PG_47', 'PG_48', 'PG_49', 'PG_50', 'PG_51', 'PG_52', 'PG_53', 'PG_54', 'PG_55', 'PG_56', 'PG_57', 'PG_58', 'PG_59', 'PG_60', 'PG_61', 'PG_62', 'PG_63', 'PG_64', 'PG_65', 'PG_66', 'PG_67', 'PG_68', 'PG_69', 'PG_70', 'PG_71', 'PG_72', 'PG_73', 'PG_74', 'PG_75', 'PG_76', 'PG_77', 'PG_78', 'PG_79', 'PG_80', 'PG_81', 'PG_82', 'PG_83', 'PG_84', 'PG_85', 'PG_86', 'PG_87', 'PG_88', 'PG_89', 'PG_90', 'PG_91', 'PG_92', 'PG_93', 'PG_

In [11]:
# --- Define the Attack Region and Boundaries ---

# 1. Define Attacking Buses (1-based ID)
ATTACK_REGION_IDS = list(range(100, 113))

# 2. Load the Network
# You must ensure pandapower is installed and run this locally.
net = pp.networks.case118()

# Convert IDs to 0-based indices used by pandapower internally
attack_region_indices = [i - 1 for i in ATTACK_REGION_IDS]

boundary_bus_indices = set()


In [12]:
print("attack_region_indices:", attack_region_indices)

attack_region_indices: [99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111]


In [16]:
print(net)

This pandapower network includes the following parameter tables:
   - bus (118 elements)
   - load (99 elements)
   - gen (53 elements)
   - shunt (14 elements)
   - ext_grid (1 element)
   - line (173 elements)
   - trafo (13 elements)
   - poly_cost (54 elements)


In [18]:
print(net.line)

     name std_type  from_bus  to_bus  length_km  r_ohm_per_km  x_ohm_per_km  \
0    None     None         0       1        1.0      5.770332     19.024956   
1    None     None         0       2        1.0      2.456676      8.074656   
2    None     None         3       4        1.0      0.335174      1.519711   
3    None     None         2       4        1.0      4.589604     20.567520   
4    None     None         4       5        1.0      2.266236     10.283760   
..    ...      ...       ...     ...        ...           ...           ...   
168  None     None        26     114        1.0      3.123216     14.111604   
169  None     None       113     114        1.0      0.438012      1.980576   
170  None     None        11     116        1.0      6.265476     26.661600   
171  None     None        74     117        1.0      2.761380      9.160164   
172  None     None        75     117        1.0      3.123216     10.359936   

     c_nf_per_km  g_us_per_km   max_i_ka   df  para

In [19]:

# Iterate over all lines in the network
for index, line in net.line.iterrows():
    from_bus_idx = line.from_bus
    to_bus_idx = line.to_bus
    
    # Check if a line connects the attack region to the non-attack region
    
    from_in_SA = from_bus_idx in attack_region_indices
    to_in_SA = to_bus_idx in attack_region_indices

    print(f"Line {index}: from_bus {from_bus_idx} (in_SA={from_in_SA}) to_bus {to_bus_idx} (in_SA={to_in_SA})")
    
    # A boundary condition is met if one end is IN the attack region (in_SA)
    # and the other end is OUTSIDE the attack region (not in_SA).
    
    if from_in_SA and not to_in_SA:
        # 'from' bus is in SA, 'to' bus is outside (Boundary Bus found at 'from')
        boundary_bus_indices.add(from_bus_idx)
        
    elif to_in_SA and not from_in_SA:
        # 'to' bus is in SA, 'from' bus is outside (Boundary Bus found at 'to')
        boundary_bus_indices.add(to_bus_idx)



Line 0: from_bus 0 (in_SA=False) to_bus 1 (in_SA=False)
Line 1: from_bus 0 (in_SA=False) to_bus 2 (in_SA=False)
Line 2: from_bus 3 (in_SA=False) to_bus 4 (in_SA=False)
Line 3: from_bus 2 (in_SA=False) to_bus 4 (in_SA=False)
Line 4: from_bus 4 (in_SA=False) to_bus 5 (in_SA=False)
Line 5: from_bus 5 (in_SA=False) to_bus 6 (in_SA=False)
Line 6: from_bus 7 (in_SA=False) to_bus 8 (in_SA=False)
Line 7: from_bus 8 (in_SA=False) to_bus 9 (in_SA=False)
Line 8: from_bus 3 (in_SA=False) to_bus 10 (in_SA=False)
Line 9: from_bus 4 (in_SA=False) to_bus 10 (in_SA=False)
Line 10: from_bus 10 (in_SA=False) to_bus 11 (in_SA=False)
Line 11: from_bus 1 (in_SA=False) to_bus 11 (in_SA=False)
Line 12: from_bus 2 (in_SA=False) to_bus 11 (in_SA=False)
Line 13: from_bus 6 (in_SA=False) to_bus 11 (in_SA=False)
Line 14: from_bus 10 (in_SA=False) to_bus 12 (in_SA=False)
Line 15: from_bus 11 (in_SA=False) to_bus 13 (in_SA=False)
Line 16: from_bus 12 (in_SA=False) to_bus 14 (in_SA=False)
Line 17: from_bus 13 (in_SA=

In [14]:
# Convert the 0-based indices back to 1-based IDs for reporting
boundary_bus_ids = [i + 1 for i in sorted(list(boundary_bus_indices))]

print(f"Attack Region (S_A): Buses {ATTACK_REGION_IDS}")
print(f"Non-Attack Region (S_N): All other buses (1-99 and 113-118)")
print(f"Boundary Buses connecting S_A to S_N: {boundary_bus_ids}")



Attack Region (S_A): Buses [100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112]
Non-Attack Region (S_N): All other buses (1-99 and 113-118)
Boundary Buses connecting S_A to S_N: [100, 102]


In [29]:
tie_lines_info = {}

In [30]:
tie_lines_info = {}

for index, line in net.line.iterrows(): 
    from_bus_id = line.from_bus 
    to_bus_id = line.to_bus 

    from_bus_in_attack_region = from_bus_id in attack_region_indices
    to_bus_in_attack_region = to_bus_id in attack_region_indices

    # Check for a tie line (one end in SA, one end in SN)
    if from_bus_in_attack_region != to_bus_in_attack_region: 
        
        if from_bus_in_attack_region:
            # Line is (Boundary Bus) -> (External Bus)
            boundary_bus = from_bus_id
            external_bus = to_bus_id
            # Flow to subtract is p_from_mw
            flow_key = 'p_from_mw'
        else:
            # Line is (External Bus) -> (Boundary Bus)
            boundary_bus = to_bus_id
            external_bus = from_bus_id
            # Flow to subtract is p_to_mw (flow at the 'to' end)
            flow_key = 'p_to_mw'
            
        # Store all necessary info for Equating/Restoring
        tie_lines_info[index] = {
            'boundary_bus_idx': boundary_bus,
            'external_bus_idx': external_bus,
            'flow_key': flow_key # Tells us where to read the flow from the results
        }
        
print(tie_lines_info)

{141: {'boundary_bus_idx': 99, 'external_bus_idx': 91, 'flow_key': 'p_to_mw'}, 142: {'boundary_bus_idx': 99, 'external_bus_idx': 93, 'flow_key': 'p_to_mw'}, 145: {'boundary_bus_idx': 99, 'external_bus_idx': 97, 'flow_key': 'p_to_mw'}, 146: {'boundary_bus_idx': 99, 'external_bus_idx': 98, 'flow_key': 'p_to_mw'}, 148: {'boundary_bus_idx': 101, 'external_bus_idx': 91, 'flow_key': 'p_to_mw'}}


In [152]:
internal_lines_info = {}

for index, line in net.line.iterrows():
    from_bus_id = line.from_bus
    to_bus_id = line.to_bus

    # Check if both buses are in attack region
    if from_bus_id in attack_region_indices and to_bus_id in attack_region_indices:
        # Store info: both buses, line index, and which side to use for flow
        # We'll use 'from_bus' side as standard
        internal_lines_info[index] = {
            'from_bus_idx': from_bus_id,
            'to_bus_idx': to_bus_id,
            'p_key': 'p_from_mw',   # flow to read from res_line
            'q_key': 'q_from_mvar'
        }

# Example output
print(internal_lines_info)


{147: {'from_bus_idx': 99, 'to_bus_idx': 100, 'p_key': 'p_from_mw', 'q_key': 'q_from_mvar'}, 149: {'from_bus_idx': 100, 'to_bus_idx': 101, 'p_key': 'p_from_mw', 'q_key': 'q_from_mvar'}, 150: {'from_bus_idx': 99, 'to_bus_idx': 102, 'p_key': 'p_from_mw', 'q_key': 'q_from_mvar'}, 151: {'from_bus_idx': 99, 'to_bus_idx': 103, 'p_key': 'p_from_mw', 'q_key': 'q_from_mvar'}, 152: {'from_bus_idx': 102, 'to_bus_idx': 103, 'p_key': 'p_from_mw', 'q_key': 'q_from_mvar'}, 153: {'from_bus_idx': 102, 'to_bus_idx': 104, 'p_key': 'p_from_mw', 'q_key': 'q_from_mvar'}, 154: {'from_bus_idx': 99, 'to_bus_idx': 105, 'p_key': 'p_from_mw', 'q_key': 'q_from_mvar'}, 155: {'from_bus_idx': 103, 'to_bus_idx': 104, 'p_key': 'p_from_mw', 'q_key': 'q_from_mvar'}, 156: {'from_bus_idx': 104, 'to_bus_idx': 105, 'p_key': 'p_from_mw', 'q_key': 'q_from_mvar'}, 157: {'from_bus_idx': 104, 'to_bus_idx': 106, 'p_key': 'p_from_mw', 'q_key': 'q_from_mvar'}, 158: {'from_bus_idx': 104, 'to_bus_idx': 107, 'p_key': 'p_from_mw', 'q_ke

In [32]:
print(f"Attack Region (S_A): Buses {attack_region_indices}")
print(f"Boundary Buses connecting S_A to S_N: {boundary_bus_indices}")
print("Tie Lines information: \n",tie_lines_info)

Attack Region (S_A): Buses [99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111]
Boundary Buses connecting S_A to S_N: {99, 101}
Tie Lines information: 
 {141: {'boundary_bus_idx': 99, 'external_bus_idx': 91, 'flow_key': 'p_to_mw'}, 142: {'boundary_bus_idx': 99, 'external_bus_idx': 93, 'flow_key': 'p_to_mw'}, 145: {'boundary_bus_idx': 99, 'external_bus_idx': 97, 'flow_key': 'p_to_mw'}, 146: {'boundary_bus_idx': 99, 'external_bus_idx': 98, 'flow_key': 'p_to_mw'}, 148: {'boundary_bus_idx': 101, 'external_bus_idx': 91, 'flow_key': 'p_to_mw'}}


In [111]:
selected_columns = [col for col in df_raw.columns if col.startswith(('PG', 'PL','QG','QL'))]

In [112]:
selected_columns

['PG1',
 'PG4',
 'PG6',
 'PG8',
 'PG10',
 'PG12',
 'PG15',
 'PG18',
 'PG19',
 'PG24',
 'PG25',
 'PG26',
 'PG27',
 'PG31',
 'PG32',
 'PG34',
 'PG36',
 'PG40',
 'PG42',
 'PG46',
 'PG49',
 'PG54',
 'PG55',
 'PG56',
 'PG59',
 'PG61',
 'PG62',
 'PG65',
 'PG66',
 'PG69',
 'PG70',
 'PG72',
 'PG73',
 'PG74',
 'PG76',
 'PG77',
 'PG80',
 'PG85',
 'PG87',
 'PG89',
 'PG90',
 'PG91',
 'PG92',
 'PG99',
 'PG100',
 'PG103',
 'PG104',
 'PG105',
 'PG107',
 'PG110',
 'PG111',
 'PG112',
 'PG113',
 'PG116',
 'PL2',
 'PL3',
 'PL5',
 'PL7',
 'PL9',
 'PL11',
 'PL13',
 'PL14',
 'PL16',
 'PL17',
 'PL20',
 'PL21',
 'PL22',
 'PL23',
 'PL28',
 'PL29',
 'PL30',
 'PL33',
 'PL35',
 'PL37',
 'PL38',
 'PL39',
 'PL41',
 'PL43',
 'PL44',
 'PL45',
 'PL47',
 'PL48',
 'PL50',
 'PL51',
 'PL52',
 'PL53',
 'PL57',
 'PL58',
 'PL60',
 'PL63',
 'PL64',
 'PL67',
 'PL68',
 'PL71',
 'PL75',
 'PL78',
 'PL79',
 'PL81',
 'PL82',
 'PL83',
 'PL84',
 'PL86',
 'PL88',
 'PL93',
 'PL94',
 'PL95',
 'PL96',
 'PL97',
 'PL98',
 'PL101',
 'PL102'

In [113]:
df_z=df_raw[selected_columns].copy()

In [114]:
net_injection_df = pd.DataFrame()

for bus_id in range(0,118):
    # Define column names
    pg_col = f"PG{bus_id+1}"
    pl_col = f"PL{bus_id+1}"
    qg_col = f"QG{bus_id+1}"
    ql_col = f"QL{bus_id+1}"

    # Define new result column names
    net_p_col = f"P{bus_id+1}"
    net_q_col = f"Q{bus_id+1}"

    # Compute safely: if the column doesn't exist, treat as 0
    PG = df_z[pg_col] if pg_col in df_z.columns else 0
    PL = df_z[pl_col] if pl_col in df_z.columns else 0
    QG = df_z[qg_col] if qg_col in df_z.columns else 0
    QL = df_z[ql_col] if ql_col in df_z.columns else 0

    # Store results in the new DataFrame
    net_injection_df[net_p_col] = PG - PL
    net_injection_df[net_q_col] = QG - QL



C:\Users\digan\AppData\Local\Temp\ipykernel_45664\3256208437.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  net_injection_df[net_p_col] = PG - PL
C:\Users\digan\AppData\Local\Temp\ipykernel_45664\3256208437.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  net_injection_df[net_q_col] = QG - QL
C:\Users\digan\AppData\Local\Temp\ipykernel_45664\3256208437.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joinin

In [115]:
df = net_injection_df.reindex(sorted(net_injection_df.columns), axis=1)

df.head()


,P1,P10,P100,P101,P102,P103,P104,P105,P106,P107,...,Q90,Q91,Q92,Q93,Q94,Q95,Q96,Q97,Q98,Q99
0,15.034523,67.342367,23.152719,-42.680052,-3.395291,42.055892,35.279180,36.871944,-18.890613,-0.589134,...,0,8.121716,7.189048,0,-18.885774,-34.109117,-14.651258,-9.735983,-7.783353,15.978392
1,21.233547,65.620240,18.079740,-45.387527,-2.360245,43.831887,33.279386,39.244785,-19.568329,-0.348296,...,0,11.780296,6.487790,0,-15.487818,-33.855608,-14.002639,-8.566263,-8.273391,21.054816
2,15.993831,79.007732,19.009790,-43.167438,-2.393326,37.390865,28.495741,37.927791,-19.647762,0.398125,...,0,9.336756,7.869400,0,-16.505178,-29.361080,-15.561053,-10.070164,-8.795248,19.584555
3,19.997194,75.310631,18.351643,-45.581452,-2.894409,40.627043,35.711135,36.303547,-21.822872,-0.383462,...,0,11.705590,5.665513,0,-17.041160,-29.745946,-17.778919,-10.504156,-8.875640,18.856928
4,15.253134,56.957230,21.704619,-38.793876,-3.062345,37.577704,29.810918,40.331491,-27.524704,0.555818,...,0,9.122052,6.895168,0,-16.562159,-32.061633,-16.702946,-9.352331,-8.820403,21.045563


In [116]:
df.columns

Index(['P1', 'P10', 'P100', 'P101', 'P102', 'P103', 'P104', 'P105', 'P106',
       'P107',
       ...
       'Q90', 'Q91', 'Q92', 'Q93', 'Q94', 'Q95', 'Q96', 'Q97', 'Q98', 'Q99'],
      dtype='object', length=236)

In [117]:
import re
import pandas as pd

def sort_by_power_type_and_number(column_name):
    """
    Creates a two-part key for sorting: (Type, Bus ID).
    1. Type: 0 for 'P' (Active Power), 1 for 'Q' (Reactive Power).
    2. Bus ID: The extracted integer (1, 2, 3...).
    """
    
    # 1. Extract Type Key (P=0, Q=1)
    power_type = column_name[0]
    type_key = 0 if power_type == 'P' else 1
    
    # 2. Extract Numerical Bus ID
    # Pattern: [P, Q] followed by one or more digits (\d+)
    match = re.search(r'[PQ](\d+)', column_name)
    bus_number = int(match.group(1)) if match else float('inf')

    # The resulting tuple (0, 1), (0, 2), ..., (1, 1), (1, 2), ... guarantees the correct order.
    return (type_key, bus_number)

# --- Applying the Sort to Your DataFrame ---

# 1. Get the current column order
current_columns = df.columns.tolist()

# 2. Sort the column list using the custom key
# This will result in: [P1, P2, P3, ..., Q1, Q2, Q3, ...]
sorted_columns = sorted(current_columns, key=sort_by_power_type_and_number)

# 3. Apply the new column order to your DataFrame
df = df.reindex(columns=sorted_columns)

print("Columns sorted successfully (P first, then Q, both numerically).")
print(df.columns)

Columns sorted successfully (P first, then Q, both numerically).
Index(['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10',
       ...
       'Q109', 'Q110', 'Q111', 'Q112', 'Q113', 'Q114', 'Q115', 'Q116', 'Q117',
       'Q118'],
      dtype='object', length=236)


In [118]:
for column in df.columns:
    print(column)

P1
P2
P3
P4
P5
P6
P7
P8
P9
P10
P11
P12
P13
P14
P15
P16
P17
P18
P19
P20
P21
P22
P23
P24
P25
P26
P27
P28
P29
P30
P31
P32
P33
P34
P35
P36
P37
P38
P39
P40
P41
P42
P43
P44
P45
P46
P47
P48
P49
P50
P51
P52
P53
P54
P55
P56
P57
P58
P59
P60
P61
P62
P63
P64
P65
P66
P67
P68
P69
P70
P71
P72
P73
P74
P75
P76
P77
P78
P79
P80
P81
P82
P83
P84
P85
P86
P87
P88
P89
P90
P91
P92
P93
P94
P95
P96
P97
P98
P99
P100
P101
P102
P103
P104
P105
P106
P107
P108
P109
P110
P111
P112
P113
P114
P115
P116
P117
P118
Q1
Q2
Q3
Q4
Q5
Q6
Q7
Q8
Q9
Q10
Q11
Q12
Q13
Q14
Q15
Q16
Q17
Q18
Q19
Q20
Q21
Q22
Q23
Q24
Q25
Q26
Q27
Q28
Q29
Q30
Q31
Q32
Q33
Q34
Q35
Q36
Q37
Q38
Q39
Q40
Q41
Q42
Q43
Q44
Q45
Q46
Q47
Q48
Q49
Q50
Q51
Q52
Q53
Q54
Q55
Q56
Q57
Q58
Q59
Q60
Q61
Q62
Q63
Q64
Q65
Q66
Q67
Q68
Q69
Q70
Q71
Q72
Q73
Q74
Q75
Q76
Q77
Q78
Q79
Q80
Q81
Q82
Q83
Q84
Q85
Q86
Q87
Q88
Q89
Q90
Q91
Q92
Q93
Q94
Q95
Q96
Q97
Q98
Q99
Q100
Q101
Q102
Q103
Q104
Q105
Q106
Q107
Q108
Q109
Q110
Q111
Q112
Q113
Q114
Q115
Q116
Q117
Q118


In [119]:
df.head()

,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10,...,Q109,Q110,Q111,Q112,Q113,Q114,Q115,Q116,Q117,Q118
0,15.034523,-45.705953,-36.470851,-0.174303,-32.642303,17.497928,-43.277044,-1.289762,-24.028900,67.342367,...,-2.965907,0,11.958772,0,2.737770,-28.242583,-7.588337,7.143427,0,-12.676726
1,21.233547,-61.063993,-37.037201,-0.118043,-35.050760,14.130329,-50.252317,-0.462877,-25.140450,65.620240,...,-2.550150,0,12.488587,0,2.750280,-26.192315,-8.059783,6.580082,0,-12.091728
2,15.993831,-50.741656,-45.892376,-1.419390,-34.251748,15.303096,-54.544774,-1.902829,-30.245074,79.007732,...,-2.555278,0,14.976653,0,2.722859,-34.729732,-7.687302,8.436127,0,-13.170712
3,19.997194,-47.499605,-30.754508,-0.234932,-32.804944,14.785502,-54.901613,-0.484383,-22.685195,75.310631,...,-2.957873,0,13.973279,0,2.793145,-32.247213,-6.163560,8.687274,0,-12.226554
4,15.253134,-56.653996,-37.101937,-0.314535,-32.824983,21.855125,-57.540147,-0.845536,-33.015502,56.957230,...,-2.595560,0,10.754824,0,2.948559,-24.447603,-7.018658,7.641694,0,-17.977146


In [120]:
import pandapower as pp
import pandapower.networks as pn

# Load the IEEE 118-bus system
net = pn.case118()


In [121]:
# Set all loads and gens to zero (so you can inject your own P and Q)
net.load['p_mw'] = 0
net.load['q_mvar'] = 0
net.gen['p_mw'] = 0
net.gen['vm_pu'] = 1.0  # keep voltage magnitude


In [122]:
row = df.iloc[0]

In [123]:
for bus in net.bus.index:
    P = row[f'P{bus+1}']   # MW
    Q = row[f'Q{bus+1}']   # Mvar

    # You can model these as loads (negative P means generation)
    pp.create_load(net, bus=bus, p_mw=max(P, 0), q_mvar=max(Q, 0))
    pp.create_sgen(net, bus=bus, p_mw=-min(P, 0), q_mvar=-min(Q, 0))

In [124]:
pp.runpp(net)


numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


In [127]:
print(net)

This pandapower network includes the following parameter tables:
   - bus (118 elements)
   - load (217 elements)
   - sgen (118 elements)
   - gen (53 elements)
   - shunt (14 elements)
   - ext_grid (1 element)
   - line (173 elements)
   - trafo (13 elements)
   - poly_cost (54 elements)
 and the following results tables:
   - res_bus (118 elements)
   - res_line (173 elements)
   - res_trafo (13 elements)
   - res_ext_grid (1 element)
   - res_load (217 elements)
   - res_sgen (118 elements)
   - res_shunt (14 elements)
   - res_gen (53 elements)


In [131]:
net.line.columns

Index(['name', 'std_type', 'from_bus', 'to_bus', 'length_km', 'r_ohm_per_km',
       'x_ohm_per_km', 'c_nf_per_km', 'g_us_per_km', 'max_i_ka', 'df',
       'parallel', 'type', 'in_service', 'max_loading_percent', 'geo'],
      dtype='object')

In [128]:
print(net.res_bus)

        vm_pu  va_degree        p_mw      q_mvar
0    1.000000  70.108470   15.000000   26.853221
1    1.017853  70.925488  -45.705953  -31.923375
2    1.005017  69.935985  -36.470851   -9.353314
3    1.000000  67.518660   -0.174303   -4.890109
4    0.999536  67.335270  -32.642303   28.388857
..        ...        ...         ...         ...
113  1.017698  57.924965  -46.213300  -28.242583
114  1.016715  57.899538  -21.621274   -7.588337
115  1.000000  36.717010   18.000000 -276.067525
116  1.029890  82.733788 -160.056248    0.000000
117  1.011101  30.743444  -30.321326  -12.676726

[118 rows x 4 columns]


In [130]:
net.res_line.columns

Index(['p_from_mw', 'q_from_mvar', 'p_to_mw', 'q_to_mvar', 'pl_mw', 'ql_mvar',
       'i_from_ka', 'i_to_ka', 'i_ka', 'vm_from_pu', 'va_from_degree',
       'vm_to_pu', 'va_to_degree', 'loading_percent'],
      dtype='object')

In [134]:
# Merge line topology (from/to bus) with results
line_flows = net.line[['from_bus', 'to_bus']].join(
    net.res_line[['p_from_mw', 'q_from_mvar', 'p_to_mw', 'q_to_mvar']]
)

print(line_flows)


     from_bus  to_bus   p_from_mw  q_from_mvar     p_to_mw     q_to_mvar
0           0       1  -18.239151   -13.504921   18.385306  1.140105e+01
1           0       2    3.239151   -13.348299   -3.216638  1.233485e+01
2           3       4   39.466570    -2.936680  -39.439015  2.851715e+00
3           2       4   41.491291    -4.634932  -41.078091  3.633651e+00
4           4       5  -48.452729     9.803635   48.745532 -9.900287e+00
..        ...     ...         ...          ...         ...           ...
168        26     114  -27.687871   -17.203596   27.856730  1.596131e+01
169       113     114    6.237829     8.098124   -6.235457 -8.372977e+00
170        11     116 -152.108924    30.129793  160.056248  3.742420e-14
171        74     117    3.556120    -3.909052   -3.552777  2.696683e+00
172        75     117  -33.671755   -10.679977   33.874102  9.980043e+00

[173 rows x 6 columns]


In [135]:
tie_lines_info

{141: {'boundary_bus_idx': 99, 'external_bus_idx': 91, 'flow_key': 'p_to_mw'},
 142: {'boundary_bus_idx': 99, 'external_bus_idx': 93, 'flow_key': 'p_to_mw'},
 145: {'boundary_bus_idx': 99, 'external_bus_idx': 97, 'flow_key': 'p_to_mw'},
 146: {'boundary_bus_idx': 99, 'external_bus_idx': 98, 'flow_key': 'p_to_mw'},
 148: {'boundary_bus_idx': 101, 'external_bus_idx': 91, 'flow_key': 'p_to_mw'}}

In [138]:
line_flows.iloc[141]

from_bus       91.000000
to_bus         99.000000
p_from_mw     -21.203902
q_from_mvar     3.037888
p_to_mw        21.514127
q_to_mvar      -6.345597
Name: 141, dtype: float64

In [139]:
df.iloc[0]

P1      15.034523
P2     -45.705953
P3     -36.470851
P4      -0.174303
P5     -32.642303
          ...    
Q114   -28.242583
Q115    -7.588337
Q116     7.143427
Q117     0.000000
Q118   -12.676726
Name: 0, Length: 236, dtype: float64

In [141]:
tie_power = {}
for line_idx, info in tie_lines_info.items():
    rows = line_flows.loc[line_idx]
    buses = info['boundary_bus_idx']
    
    # pick direction
    if info['flow_key'] == 'p_to_mw':
        p_flow = rows['p_to_mw']
        q_flow = rows['q_to_mvar']
    else:
        p_flow = rows['p_from_mw']
        q_flow = rows['q_from_mvar']
    
    # sum per boundary bus
    if bus not in tie_power:
        tie_power[buses] = {'P_tie': 0.0, 'Q_tie': 0.0}
    
    tie_power[buses]['P_tie'] += p_flow
    tie_power[buses]['Q_tie'] += q_flow

print(pd.DataFrame(tie_power).T)

         P_tie     Q_tie
99  -28.002765  5.472567
101  42.513959 -1.129893


In [142]:
print(tie_power)

{99: {'P_tie': np.float64(-28.002764848755916), 'Q_tie': np.float64(5.472567398742971)}, 101: {'P_tie': np.float64(42.513958908757985), 'Q_tie': np.float64(-1.1298929367423647)}}


In [151]:
data = {}
for bus, vals in tie_power.items():
    data[f'P{bus}'] = [vals['P_tie']]  # put in a list so it's a single-row DataFrame
    data[f'Q{bus}'] = [vals['Q_tie']]

tie_df = pd.DataFrame(data)
print(tie_df)

         P99       Q99       P101      Q101
0 -28.002765  5.472567  42.513959 -1.129893


In [146]:
row

P1      15.034523
P2     -45.705953
P3     -36.470851
P4      -0.174303
P5     -32.642303
          ...    
Q114   -28.242583
Q115    -7.588337
Q116     7.143427
Q117     0.000000
Q118   -12.676726
Name: 0, Length: 236, dtype: float64

In [148]:
print(row['P100'],"  ",row['P102'])
print(row['Q100'],"  ",row['Q102'])

23.15271896    -3.395290977
16.62099378    -2.719747029


In [149]:
for bus, vals in tie_power.items():
    p_bus=f'P{bus+1}'
    q_bus=f'Q{bus+1}'
    row[p_bus] -= vals['P_tie']
    row[q_bus] -= vals['Q_tie']


In [150]:
print(row['P100'],"  ",row['P102'])
print(row['Q100'],"  ",row['Q102'])

51.15548380875592    -45.90924988575799
11.148426381257028    -1.5898540922576354
